# HAN Email Spam Detection — Colab Training
**Paper**: Zavrak & Yilmaz (2023) — *Email spam detection using hierarchical
attention hybrid deep learning method*

## What this notebook does
Exactly follows the experimental protocol from Section 4.3:
1. **Same-dataset evaluation**: 10-fold stratified CV on Enron, then on SA
2. **Cross-dataset evaluation**: train on Enron → test on SA (and vice versa)

**Crash-resilient**: each fold saves its checkpoint to Drive immediately.
Re-running after a disconnect skips already-completed folds automatically.

## Before running
- Runtime → Change runtime type → **T4 GPU**
- Your preprocessed data must be in `My Drive/han-spam/data/processed/`
  (run `python scripts/drive_sync.py push-data` locally first)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/han-spam'
DATA_DIR     = f'{DRIVE_ROOT}/data/processed'
CKPT_DIR     = f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR  = f'{DRIVE_ROOT}/results'

import os
os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(DATA_DIR), (
    f"Data not found: {DATA_DIR}\n"
    "Run 'python scripts/drive_sync.py push-data' locally first."
)
print('Data dir contents:', os.listdir(DATA_DIR))

## 2. Check GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {gpus}')
if not gpus:
    print('WARNING: No GPU — Runtime → Change runtime type → T4 GPU')

## 3. Load preprocessed data + config

In [ ]:
import json, yaml
import numpy as np
from types import SimpleNamespace

def to_ns(d):
    ns = SimpleNamespace()
    for k, v in d.items():
        setattr(ns, k, to_ns(v) if isinstance(v, dict) else v)
    return ns

# Inline config — keep in sync with configs/config.yaml
CONFIG_YAML = """
model:
  cnn_filters: [64, 128]
  cnn_kernels: [3, 5]
  word_gru_units: 50
  word_attention_dim: 100
  sent_gru_units: 50
  sent_attention_dim: 100
  dropout_rate: 0.5
  output_activation: sigmoid
  use_tcn: false
  tcn_filters: 64
  tcn_kernel_size: 3
  tcn_dilations: [1, 2, 4, 8]
  tcn_dropout: 0.2
training:
  epochs: 20
  batch_size: 64
  learning_rate: 0.001
  cv_folds: 10
  early_stopping_patience: 3
project:
  seed: 42
"""
CFG = to_ns(yaml.safe_load(CONFIG_YAML))

# Load artefacts
embedding_matrix = np.load(f'{DATA_DIR}/embedding_matrix.npy')

X_en_train = np.load(f'{DATA_DIR}/X_enron_train.npy')
y_en_train = np.load(f'{DATA_DIR}/y_enron_train.npy')
X_en_test  = np.load(f'{DATA_DIR}/X_enron_test.npy')
y_en_test  = np.load(f'{DATA_DIR}/y_enron_test.npy')

X_sa_train = np.load(f'{DATA_DIR}/X_sa_train.npy')
y_sa_train = np.load(f'{DATA_DIR}/y_sa_train.npy')
X_sa_test  = np.load(f'{DATA_DIR}/X_sa_test.npy')
y_sa_test  = np.load(f'{DATA_DIR}/y_sa_test.npy')

with open(f'{DATA_DIR}/manifest.json') as f:
    manifest = json.load(f)

vocab_size    = embedding_matrix.shape[0]
embed_dim     = embedding_matrix.shape[1]
max_sentences = manifest['max_sentences']
max_words     = manifest['max_words']

print('embedding_matrix:', embedding_matrix.shape)
print('X_enron_train:', X_en_train.shape, 'y:', y_en_train.shape)
print('X_sa_train   :', X_sa_train.shape, 'y:', y_sa_train.shape)
print('manifest:', json.dumps(manifest, indent=2))

## 4. HAN model definition
Implements Fig. 1 and Fig. 2 from the paper exactly:
FastText embedding → CNN (multi-filter) → BiGRU → word attention →
sentence BiGRU → sentence attention → Dropout → Dense(sigmoid)

In [ ]:
import tensorflow.keras.backend as K
from tensorflow.keras import layers, models, initializers


class AttentionLayer(layers.Layer):
    """
    Additive attention (Yang et al. 2016 / Bahdanau et al. 2015).
    Used at both word-level and sentence-level (paper Eq. 5-7 and 10-12).
    """
    def __init__(self, attention_dim, **kwargs):
        super().__init__(**kwargs)
        self.attention_dim = attention_dim

    def build(self, input_shape):
        d = input_shape[-1]
        self.W = self.add_weight(name='W', shape=(d, self.attention_dim),
                                 initializer=initializers.GlorotUniform())
        self.b = self.add_weight(name='b', shape=(self.attention_dim,),
                                 initializer='zeros')
        self.u = self.add_weight(name='u', shape=(self.attention_dim, 1),
                                 initializer=initializers.GlorotUniform())
        super().build(input_shape)

    def call(self, x, mask=None):
        # uit = tanh(W*hit + b)  [Eq. 5 / 10]
        uit = K.tanh(K.dot(x, self.W) + self.b)
        # ait = softmax(uit . u)  [Eq. 6 / 11]
        ait = K.squeeze(K.dot(uit, self.u), -1)
        ait = K.exp(ait)
        if mask is not None:
            ait *= K.cast(mask, K.floatx())
        ait /= (K.sum(ait, axis=1, keepdims=True) + K.epsilon())
        # si = sum(ait * hit)  [Eq. 7 / 12]
        return K.sum(x * K.expand_dims(ait), axis=1)

    def compute_mask(self, inputs, mask=None):
        return None   # absorbs the mask — no mask propagated further


def build_word_encoder(max_words, vocab_size, embed_dim, embedding_matrix, cfg):
    """Word-level: Embedding -> CNN bank -> BiGRU -> Attention."""
    inp = layers.Input(shape=(max_words,), dtype='int32')

    # Embedding (FastText weights, frozen per paper)
    x = layers.Embedding(
        vocab_size, embed_dim,
        weights=[embedding_matrix],
        mask_zero=True,
        trainable=False,
    )(inp)

    # Detach mask before CNN — Conv1D doesn't support masking in Keras 3.x
    x = layers.Lambda(lambda t: t, output_shape=lambda s: s)(x)

    # Multi-filter CNN (paper Fig.2: filters [64,128], kernels [3,5])
    conv_outs = []
    for n_filters, k_size in zip(cfg.model.cnn_filters, cfg.model.cnn_kernels):
        c = layers.Conv1D(n_filters, k_size, padding='same', activation='relu')(x)
        conv_outs.append(c)
    cnn = layers.Concatenate()(conv_outs) if len(conv_outs) > 1 else conv_outs[0]

    # BiGRU word encoder (paper: 50 units each direction -> 100 concat)
    h = layers.Bidirectional(
        layers.GRU(cfg.model.word_gru_units, return_sequences=True)
    )(cnn)

    # Word attention -> sentence vector
    sent_vec = AttentionLayer(cfg.model.word_attention_dim)(h)

    return models.Model(inp, sent_vec, name='word_encoder')


def build_han(max_sentences, max_words, vocab_size, embed_dim, embedding_matrix, cfg):
    """Full HAN: [doc: max_sentences x max_words] -> binary spam probability."""
    word_enc = build_word_encoder(max_words, vocab_size, embed_dim, embedding_matrix, cfg)

    doc_inp = layers.Input(shape=(max_sentences, max_words), dtype='int32')

    # Encode each sentence independently (shared weights)
    sent_vecs = layers.TimeDistributed(word_enc)(doc_inp)

    # BiGRU sentence encoder (50 units -> 100 concat)
    h = layers.Bidirectional(
        layers.GRU(cfg.model.sent_gru_units, return_sequences=True)
    )(sent_vecs)

    # Sentence attention -> document vector
    doc_vec = AttentionLayer(cfg.model.sent_attention_dim)(h)

    # Classifier head
    out = layers.Dropout(cfg.model.dropout_rate)(doc_vec)
    out = layers.Dense(1, activation=cfg.model.output_activation)(out)

    model = models.Model(doc_inp, out, name='HAN')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(cfg.training.learning_rate),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model

print('Model builder ready.')
model = build_han(max_sentences, max_words, vocab_size, embed_dim, embedding_matrix, CFG)
model.summary()

## 5. Training helpers

In [ ]:
import time
import pandas as pd
from sklearn.model_selection import StratifiedKFold


def run_cv(X, y, dataset_name, cfg, resume=True):
    """
    10-fold stratified CV on one dataset.
    Crash-resilient: each fold's checkpoint + metrics saved to Drive immediately.
    Resumable: skips folds whose checkpoint already exists.

    Returns a list of per-fold metric dicts.
    """
    metrics_path = f'{RESULTS_DIR}/{dataset_name}_cv_metrics.json'
    fold_results = []

    if resume and os.path.exists(metrics_path):
        with open(metrics_path) as f:
            fold_results = json.load(f)
        print(f'[{dataset_name}] Resuming — {len(fold_results)} folds already done')

    completed = {r['fold'] for r in fold_results}

    skf = StratifiedKFold(n_splits=cfg.training.cv_folds,
                          shuffle=True, random_state=cfg.project.seed)

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        fold_num = fold_idx + 1
        ckpt = f'{CKPT_DIR}/{dataset_name}_fold_{fold_num:02d}.weights.h5'

        if fold_num in completed and os.path.exists(ckpt):
            print(f'  fold {fold_num:2d}/10 already done — skipping')
            continue

        print(f'\n{"="*55}')
        print(f'[{dataset_name}] fold {fold_num}/10')
        print(f'{"="*55}')

        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        tf.keras.backend.clear_session()
        model = build_han(max_sentences, max_words,
                          vocab_size, embed_dim, embedding_matrix, cfg)

        t0 = time.time()
        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=cfg.training.epochs,
            batch_size=cfg.training.batch_size,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_auc', mode='max',
                    patience=cfg.training.early_stopping_patience,
                    restore_best_weights=True,
                ),
                tf.keras.callbacks.ModelCheckpoint(
                    ckpt, monitor='val_auc', mode='max',
                    save_best_only=True, save_weights_only=True,
                ),
            ],
            verbose=2,
        )
        elapsed = time.time() - t0

        # Ensure checkpoint is written even if EarlyStopping fired on epoch 1
        model.save_weights(ckpt)

        val_m = model.evaluate(X_val, y_val, verbose=0, return_dict=True)
        result = {
            'fold': fold_num,
            'elapsed_sec': round(elapsed, 1),
            'epochs': len(history.history['loss']),
            **{k: round(float(v), 6) for k, v in val_m.items()},
        }
        fold_results.append(result)

        # Persist immediately -> crash safety
        with open(metrics_path, 'w') as f:
            json.dump(fold_results, f, indent=2)

        print(f'  done in {elapsed:.0f}s | '
              f'val_auc={result.get("auc", 0):.4f} '
              f'val_acc={result.get("accuracy", 0):.4f}')

    return fold_results


def print_cv_summary(fold_results, dataset_name):
    df = pd.DataFrame(fold_results)
    print(f'\n{"─"*55}')
    print(f'{dataset_name} — 10-fold CV summary')
    print(f'{"─"*55}')
    for m in ['auc', 'accuracy', 'precision', 'recall']:
        if m in df.columns:
            print(f'  {m:<12} {df[m].mean():.4f} ± {df[m].std():.4f}')
    return df


def best_checkpoint(fold_results, dataset_name):
    df = pd.DataFrame(fold_results)
    best_fold = int(df.loc[df['auc'].idxmax(), 'fold'])
    return f'{CKPT_DIR}/{dataset_name}_fold_{best_fold:02d}.weights.h5', best_fold


def evaluate_on(ckpt_path, X_test, y_test, label=''):
    tf.keras.backend.clear_session()
    m = build_han(max_sentences, max_words,
                  vocab_size, embed_dim, embedding_matrix, CFG)
    m.load_weights(ckpt_path)
    metrics = m.evaluate(X_test, y_test, verbose=0, return_dict=True)
    print(f'\n{label}')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')
    return {k: float(v) for k, v in metrics.items()}

## 6. Same-dataset evaluation — Enron (10-fold CV)

In [ ]:
en_results = run_cv(X_en_train, y_en_train, 'enron', CFG)
en_df = print_cv_summary(en_results, 'Enron')

## 7. Same-dataset evaluation — SpamAssassin (10-fold CV)

In [ ]:
sa_results = run_cv(X_sa_train, y_sa_train, 'spamassassin', CFG)
sa_df = print_cv_summary(sa_results, 'SpamAssassin')

## 8. Cross-dataset evaluation
Matches Table 3 of the paper:
  - Train on Enron best fold → test on SA
  - Train on SA best fold → test on Enron

In [ ]:
en_ckpt, en_best_fold = best_checkpoint(en_results, 'enron')
sa_ckpt, sa_best_fold = best_checkpoint(sa_results, 'spamassassin')

print(f'Best Enron fold: {en_best_fold}  ->  {en_ckpt}')
print(f'Best SA fold:    {sa_best_fold}  ->  {sa_ckpt}')

# Train on Enron, test on SA
cd_en_to_sa = evaluate_on(en_ckpt, X_sa_test, y_sa_test,
                           'Cross-dataset: train=Enron, test=SA')

# Train on SA, test on Enron
cd_sa_to_en = evaluate_on(sa_ckpt, X_en_test, y_en_test,
                           'Cross-dataset: train=SA, test=Enron')

# Same-dataset held-out test
sd_en = evaluate_on(en_ckpt, X_en_test, y_en_test,
                    'Same-dataset held-out: Enron')
sd_sa = evaluate_on(sa_ckpt, X_sa_test, y_sa_test,
                    'Same-dataset held-out: SA')

## 9. Save all results

In [ ]:
all_results = {
    'enron_cv':    en_results,
    'sa_cv':       sa_results,
    'cross_en_to_sa': cd_en_to_sa,
    'cross_sa_to_en': cd_sa_to_en,
    'held_out_en': sd_en,
    'held_out_sa': sd_sa,
}
with open(f'{RESULTS_DIR}/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('\nAll results saved to Drive.')
print(f'  {RESULTS_DIR}/all_results.json')
print(f'  {RESULTS_DIR}/enron_cv_metrics.json')
print(f'  {RESULTS_DIR}/spamassassin_cv_metrics.json')
print('\nPull locally with:')
print('  python scripts/drive_sync.py pull-checkpoints')
print('  python scripts/drive_sync.py pull-results')